# 2. PLSSVD evaluation

Show unpermuted results first, followed by the fixed full-data PCA comparison, permutation effects, and component selection. Training-learned transformations are applied to held-out trials from the same participants.

`N_ITERATIONS=None` uses all prepared iterations; an integer selects the first N completed, prepared iterations. Summary figures include every fold in that selection. Means and sample SD describe split variability, not independent replicates or confidence intervals. Permutation comparisons and component selection retain their existing matched-iteration rules and use all matched iterations.


In [ ]:
from pathlib import Path
import sys
import json
import shlex
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from IPython.display import display

ROOT = Path.cwd()
MEG_KIND = 'paired_coverage'
BLOCK_SCALING = True
N_COMPONENTS = 10
PCA_ABSOLUTE = True
N_ITERATIONS = None
SELECTION_PERM = 'both'
PERMUTATION_TYPE = 'time_block'
N_FIT_ITERATIONS = 100
N_SPLITS = 5
MAX_COMPONENTS = 100
SEED = 2026
CACHE_DIR = ROOT / 'out' / 'trial_cache'
RUNS_DIR = ROOT / 'out' / ('plssvd_eval_equal_var' if BLOCK_SCALING else 'plssvd_eval') / MEG_KIND
SCALING_MODE = 'equal_variance' if BLOCK_SCALING else 'none'
PARTITION_COLORS = {'train': '#555555', 'test': '#9467BD'}

sys.path.insert(0, str(ROOT))
from vs1_coverage_figures import apply_style, MODALITY_COLORS
from vs2_1_plssvd_results import load_summary
from vs2_1_plssvd_fit_pca import load_comparison as load_full_pca_comparison
from vs2_1_plssvd_figures import load_comparison
from vs2_1_plssvd_select_components import load_selection
from vs2_1_plssvd_eval_utils import validation_run_name
apply_style()


## Unpermuted results: reconstruction, prediction, and shared covariance

Reconstruction fractions measure retained own-modality signal. Prediction Q² compares cross-modal predictions against the training-mean baseline and may be negative. Cross-covariance energy is the retained squared Frobenius norm divided by full cross-covariance energy. The performance plot reports the mean **signed** paired Pearson r; correlation heatmaps report mean **absolute** r.

Covariance stays signed. Every summary uses all selected fold–iteration results with equal weight. Time courses retain native component rank, with both modalities oriented by the largest absolute training iEEG score and divided by their respective training score SD. Signs are never chosen using test data; components are not rematched across folds.


In [ ]:
INPUT_DIR = RUNS_DIR / 'none'
OUTPUT_DIR = INPUT_DIR / 'summaries'
RECOMPUTE = False

fit_command = ['python', '-u', str(ROOT/'vs2_1_plssvd_eval.py'), '--root', str(ROOT),
    '--output-dir', str(INPUT_DIR), '--cache-dir', str(CACHE_DIR), '--meg-kind', MEG_KIND,
    '--block-scaling', SCALING_MODE, '--max-components', str(MAX_COMPONENTS),
    '--n-iterations', str(N_FIT_ITERATIONS), '--n-splits', str(N_SPLITS), '--seed', str(SEED),
    '--perm', 'none', '--perm-type', 'none']
post_command = ['python', '-u', str(ROOT/'vs2_1_plssvd_postprocess.py'), '--runs-dir', str(RUNS_DIR),
    '--components', str(N_COMPONENTS), '--perm', 'none', '--perm-type', 'none']
summary_command = ['python', '-u', str(ROOT/'vs2_1_plssvd_results.py'), '--run-dir', str(INPUT_DIR), '--components', str(N_COMPONENTS)]
if N_ITERATIONS is not None: summary_command += ['--iterations', str(N_ITERATIONS)]
print('Fit if missing:', shlex.join(fit_command))
print('Prepare evaluation:', shlex.join(post_command))
print('Prepare selected-iteration summary:', shlex.join(summary_command))
evaluation = None
if not RECOMPUTE:
    try:
        evaluation = load_summary(INPUT_DIR, N_COMPONENTS, N_ITERATIONS,
                                  meg_kind=MEG_KIND, block_scaling=SCALING_MODE)
        display(pd.Series(evaluation['metadata']['options'], name='Saved fit settings'))
        print('Selected iterations:', evaluation['metadata']['iterations'])
    except (FileNotFoundError, ValueError) as error:
        print(error)
if evaluation is None:
    print('Run the required commands, then set RECOMPUTE=False and rerun this cell.')


In [ ]:
FIGSIZE = (15, 6)
JITTER_SEED = 2026
METRICS = ['ieeg_reconstruction_fraction', 'meg_reconstruction_fraction', 'predict_ieeg_q2',
           'predict_meg_q2', 'crosscov_energy_fraction', 'mean_r']
METRIC_LABELS = ['iEEG reconstruction', 'MEG reconstruction', 'Predict iEEG Q²',
                 'Predict MEG Q²', 'Cross-covariance energy', 'Paired r score']

if evaluation is not None:
    frame = evaluation['fold_metrics']
    fig, ax = plt.subplots(figsize=FIGSIZE, constrained_layout=True)
    rng = np.random.default_rng(JITTER_SEED)
    for index, metric in enumerate(METRICS):
        for part, side in [('train', -1), ('test', 1)]:
            values = frame.loc[frame.partition == part, metric].dropna().to_numpy()
            values = values[np.isfinite(values)]
            if not len(values): continue
            if len(values) > 1 and np.ptp(values) > 0:
                grid = np.linspace(values.min(), values.max(), 200)
                density = gaussian_kde(values)(grid)
                width = .35*density/density.max()
                ax.fill_betweenx(grid, index, index+side*width, color=PARTITION_COLORS[part], alpha=.3)
            ax.scatter(index+side*rng.uniform(.04, .32, len(values)), values,
                       color=PARTITION_COLORS[part], s=10, alpha=.45, label=part if index == 0 else None)
            ax.plot([index, index+side*.35], [np.median(values)]*2, color=PARTITION_COLORS[part], linewidth=2)
    ax.axhline(0, color='black', linewidth=.7)
    ax.set(xticks=range(len(METRICS)), xticklabels=METRIC_LABELS, ylabel='Metric value',
           title='Performance across all selected splits and iterations')
    ax.tick_params(axis='x', labelrotation=15); ax.legend(); ax.grid(True)
    plt.show()


In [ ]:
FIGSIZE = (max(14, N_COMPONENTS*1.2), max(6, N_COMPONENTS*.6))
ANNOTATION_SIZE = max(5, min(10, 80/N_COMPONENTS))

if evaluation is not None:
    arrays = evaluation['arrays']
    bound = max(np.nanmax(abs(arrays[part+'_covariance_mean'])) for part in ('train','test')) or 1.
    fig, axes = plt.subplots(1, 2, figsize=FIGSIZE, constrained_layout=True)
    for ax, part in zip(axes, ['train','test']):
        mean, sd = arrays[part+'_covariance_mean'], arrays[part+'_covariance_std']
        im = ax.imshow(mean, cmap="RdBu_r", vmin=-bound, vmax=bound)
        for (i, j), value in np.ndenumerate(mean):
            ax.text(j, i, f'{value:.2g}\n± {sd[i,j]:.2g}', ha='center', va='center', fontsize=ANNOTATION_SIZE,
                    color='black', bbox=dict(facecolor='white', alpha=.65, edgecolor='none', pad=.1))
        ax.set(title=f'{part}: Signed score cross-covariance, mean ± SD', xlabel='MEG component', ylabel='iEEG component',
               xticks=np.arange(len(mean)), xticklabels=np.arange(1,len(mean)+1),
               yticks=np.arange(len(mean)), yticklabels=np.arange(1,len(mean)+1)); ax.grid(True)
        fig.colorbar(im, ax=ax)
    plt.show()


In [ ]:
FIGSIZE = (10, 4)

if evaluation is not None:
    arrays = evaluation['arrays']
    fig, ax = plt.subplots(figsize=FIGSIZE, constrained_layout=True)
    for part in ['train', 'test']:
        mean = np.diag(arrays[part+'_covariance_mean'])
        sd = np.diag(arrays[part+'_covariance_std'])
        ax.errorbar(np.arange(1,len(mean)+1), mean, yerr=sd, fmt='o-', capsize=3,
                    color=PARTITION_COLORS[part], label=part)
    ax.set(xlabel='Paired component', ylabel='Signed score cross-covariance', title='Diagonal values: mean ± SD')
    ax.grid(True); ax.legend(); plt.show()


In [ ]:
FIGSIZE = (max(14, N_COMPONENTS*1.2), max(6, N_COMPONENTS*.6))
ANNOTATION_SIZE = max(5, min(10, 80/N_COMPONENTS))

if evaluation is not None:
    arrays = evaluation['arrays']
    bound = 1.
    fig, axes = plt.subplots(1, 2, figsize=FIGSIZE, constrained_layout=True)
    for ax, part in zip(axes, ['train','test']):
        mean, sd = arrays[part+'_correlation_absolute_mean'], arrays[part+'_correlation_absolute_std']
        im = ax.imshow(mean, cmap="viridis", vmin=0, vmax=bound)
        for (i, j), value in np.ndenumerate(mean):
            ax.text(j, i, f'{value:.2g}\n± {sd[i,j]:.2g}', ha='center', va='center', fontsize=ANNOTATION_SIZE,
                    color='black', bbox=dict(facecolor='white', alpha=.65, edgecolor='none', pad=.1))
        ax.set(title=f'{part}: Absolute Pearson r, mean ± SD', xlabel='MEG component', ylabel='iEEG component',
               xticks=np.arange(len(mean)), xticklabels=np.arange(1,len(mean)+1),
               yticks=np.arange(len(mean)), yticklabels=np.arange(1,len(mean)+1)); ax.grid(True)
        fig.colorbar(im, ax=ax)
    plt.show()


In [ ]:
FIGSIZE = (10, 4)

if evaluation is not None:
    arrays = evaluation['arrays']
    fig, ax = plt.subplots(figsize=FIGSIZE, constrained_layout=True)
    for part in ['train', 'test']:
        mean = np.diag(arrays[part+'_correlation_absolute_mean'])
        sd = np.diag(arrays[part+'_correlation_absolute_std'])
        ax.errorbar(np.arange(1,len(mean)+1), mean, yerr=sd, fmt='o-', capsize=3,
                    color=PARTITION_COLORS[part], label=part)
    ax.set(xlabel='Paired component', ylabel='Absolute Pearson r', title='Diagonal values: mean ± SD')
    ax.grid(True); ax.legend(); plt.show()


In [ ]:
PLOT_COMPONENTS = min(3, N_COMPONENTS)
FIGSIZE = (13, 3*PLOT_COMPONENTS)

if evaluation is not None:
    arrays = evaluation['arrays']
    fig, axes = plt.subplots(PLOT_COMPONENTS, 1, figsize=FIGSIZE, squeeze=False, constrained_layout=True)
    for pc, ax in enumerate(axes[:, 0]):
        for modality, color in [('ieeg', MODALITY_COLORS['iEEG']), ('meg', MODALITY_COLORS['MEG'])]:
            for part, style in [('train', '--'), ('test', '-')]:
                key = f'{part}_{modality}_normalized'
                mean, sd = arrays[key+'_mean'][:, pc], arrays[key+'_std'][:, pc]
                ax.plot(arrays['times'], mean, color=color, linestyle=style, label=f'{modality} {part}')
                ax.fill_between(arrays['times'], mean-sd, mean+sd, color=color, alpha=.12)
        ax.set(title=f'Component {pc+1}: all selected folds and iterations', xlabel='Time (s)', ylabel='Score / training SD')
        ax.grid(True); ax.legend(ncol=4)
    plt.show()


## Comparison with fixed full-data PCA

Fit unpermuted PCA on all trials using the selected MEG construction and the saved run's random subject association. Correlate it with saved train/test PLSSVD scores, retaining folds separately. `PCA_ABSOLUTE` controls whether absolute or signed correlations are averaged. This is a descriptive full-data reference, not held-out validation of PCA.


In [ ]:
INPUT_DIR = RUNS_DIR / 'none'
OUTPUT_DIR = INPUT_DIR / 'full_pca'
RECOMPUTE = False

command = ['python', '-u', str(ROOT/'vs2_1_plssvd_fit_pca.py'), '--run-dir', str(INPUT_DIR),
           '--cache-dir', str(CACHE_DIR), '--meg-kind', MEG_KIND, '--block-scaling', SCALING_MODE,
           '--components', str(N_COMPONENTS), '--absolute' if PCA_ABSOLUTE else '--no-absolute']
if N_ITERATIONS is not None: command += ['--iterations', str(N_ITERATIONS)]
print(shlex.join(command))
full_pca_result = None
if not RECOMPUTE:
    try:
        full_pca_result = load_full_pca_comparison(INPUT_DIR, N_COMPONENTS, meg_kind=MEG_KIND,
            block_scaling=SCALING_MODE, absolute=PCA_ABSOLUTE, n_iterations=N_ITERATIONS)
    except (FileNotFoundError, ValueError) as error:
        print(error)
if full_pca_result is None:
    print('Run the command, then set RECOMPUTE=False and rerun this cell.')


In [ ]:
ANNOTATION_SIZE = max(4, min(8, 60/N_COMPONENTS))

if full_pca_result is not None:
    folds = full_pca_result['n_folds']
    k = full_pca_result['n_components']
    fig, axes = plt.subplots(4, folds, figsize=(max(4,k*.5)*folds, max(14,k*1.8)), squeeze=False, constrained_layout=True)
    for mi, modality in enumerate(['iEEG','MEG']):
        for pi, part in enumerate(['train','test']):
            for fold in range(folds):
                ax = axes[2*mi+pi, fold]
                mean, sd = full_pca_result['mean'][mi,pi,fold], full_pca_result['std'][mi,pi,fold]
                im = ax.imshow(mean, vmin=0 if PCA_ABSOLUTE else -1, vmax=1, cmap='viridis' if PCA_ABSOLUTE else 'RdBu_r')
                for (i,j), value in np.ndenumerate(mean):
                    ax.text(j,i,f'{value:.2f}\n± {sd[i,j]:.2f}',ha='center',va='center',fontsize=ANNOTATION_SIZE,
                            bbox=dict(facecolor='white',alpha=.65,edgecolor='none',pad=.1))
                ax.set(title=f'{modality} {part}, fold {fold+1}',xlabel='Full-data PCA component',ylabel='PLSSVD component',
                       xticks=np.arange(k),xticklabels=np.arange(1,k+1),yticks=np.arange(k),yticklabels=np.arange(1,k+1)); ax.grid(True)
    fig.colorbar(im, ax=axes.ravel().tolist(), shrink=.6, label='Mean |r|' if PCA_ABSOLUTE else 'Mean r')
    fig.suptitle(f'Fixed full-data PCA: {MEG_KIND}; mean ± SD across {len(full_pca_result["iterations"])} iterations')
    plt.show()


## Permutation effects across iterations

The existing matched-iteration comparison is retained. Each plotted value is a held-out fold mean within an iteration. Permutations operate before fitting; time permutations are shared across conditions and train/test within each iteration. Baseline and null runs must have compatible settings and matched splits. Distributions are descriptive, not independent replicates.


In [ ]:
INPUT_DIR = RUNS_DIR
RECOMPUTE = False
PERMUTATION_MODES = ['meg', 'ieeg', 'both']

null_comparisons = {}
for mode in PERMUTATION_MODES:
    name = validation_run_name(mode, PERMUTATION_TYPE)
    print('Fit if missing:', shlex.join(['python', '-u', str(ROOT/'vs2_1_plssvd_eval.py'), '--root', str(ROOT),
        '--output-dir', str(INPUT_DIR/name), '--cache-dir', str(CACHE_DIR), '--meg-kind', MEG_KIND,
        '--block-scaling', SCALING_MODE, '--max-components', str(MAX_COMPONENTS), '--n-iterations', str(N_FIT_ITERATIONS),
        '--n-splits', str(N_SPLITS), '--seed', str(SEED), '--perm', mode, '--perm-type', PERMUTATION_TYPE]))
    print('Prepare:', shlex.join(['python', '-u', str(ROOT/'vs2_1_plssvd_postprocess.py'), '--runs-dir', str(INPUT_DIR),
        '--components', str(N_COMPONENTS), '--perm', mode, '--perm-type', PERMUTATION_TYPE]))
    if not RECOMPUTE:
        try:
            data = load_comparison(INPUT_DIR, name, N_COMPONENTS)
            options = data['metadata']['options'] if 'options' in data['metadata'] else None
            if options is not None and (options['meg_kind'] != MEG_KIND or options['block_scaling'] != SCALING_MODE):
                raise ValueError('Saved comparison settings differ from the notebook.')
            null_comparisons[mode] = data
        except (FileNotFoundError, ValueError) as error:
            print(error)


In [ ]:
FIGSIZE = (18, 3.4*len(PERMUTATION_MODES))
PANELS = [('mean_paired_covariance','Mean paired covariance'), ('mean_r','Mean signed Pearson correlation'),
          ('ieeg_reconstruction_fraction','iEEG reconstruction fraction'), ('meg_reconstruction_fraction','MEG reconstruction fraction')]

fig, axes = plt.subplots(len(PERMUTATION_MODES), len(PANELS), figsize=FIGSIZE, squeeze=False, constrained_layout=True)
for row, mode in enumerate(PERMUTATION_MODES):
    data = null_comparisons.get(mode)
    if data is None:
        for ax in axes[row]: ax.text(.5,.5,'Comparison not prepared',ha='center',transform=ax.transAxes); ax.grid(True)
        continue
    frame = data['metrics'].query("partition == 'test'")
    for ax, (metric, title) in zip(axes[row], PANELS):
        values = {label: frame.loc[frame.run == label, metric].dropna().to_numpy() for label in ['Unpermuted','Permuted']}
        values = {label: array[np.isfinite(array)] for label,array in values.items()}
        combined = np.concatenate(list(values.values()))
        if not len(combined): continue
        bins = np.histogram_bin_edges(combined, bins=min(25,max(5,int(np.sqrt(len(combined))))))
        for label, color in [('Unpermuted','tab:blue'), ('Permuted','tab:orange')]:
            array = values[label]
            if len(array):
                ax.hist(array,bins=bins,histtype='step',linewidth=2,color=color,label=f'{label} (n={len(array)})')
                ax.axvline(np.median(array),color=color,linestyle=':',alpha=.7)
        ax.set(title=f'{mode.upper()} randomized: {title}',xlabel='Held-out fold mean per iteration',ylabel='Iterations')
        ax.legend(fontsize=8); ax.grid(True)
fig.suptitle(f'{PERMUTATION_TYPE} vs baseline; k={N_COMPONENTS}; matched iterations, dotted lines: medians')
plt.show()


## Component selection: stable excess covariance against a selected null

Unchanged rule: at each rank, take the 95th percentile of the null's iteration fold-mean **signed** covariance, floored at zero. Retain contiguous leading ranks whose baseline covariance strictly exceeds this threshold in at least 90% of matched iterations. Correlation and reconstruction remain supporting diagnostics. `N_COMPONENTS`, `SELECTION_PERM`, and `PERMUTATION_TYPE` are defined at the beginning of the notebook.


In [ ]:
INPUT_DIR = RUNS_DIR
RECOMPUTE = False

command = ['python', '-u', str(ROOT/'vs2_1_plssvd_select_components.py'), '--runs-dir', str(INPUT_DIR),
           '--components', str(N_COMPONENTS), '--perm', SELECTION_PERM, '--perm-type', PERMUTATION_TYPE]
if RECOMPUTE: command.append('--recompute')
print(shlex.join(command))
selection_result = None
if not RECOMPUTE:
    try:
        selection_result = load_selection(INPUT_DIR, N_COMPONENTS, SELECTION_PERM, PERMUTATION_TYPE)
        print('Selected components:', selection_result['summary']['selected_k'])
        display(selection_result['selection'])
    except (FileNotFoundError, ValueError) as error:
        print(error)


In [ ]:
result = selection_result

if result is not None:
    import matplotlib.pyplot as plt
    table, summary = result['selection'], result['summary']
    fig, axes = plt.subplots(2,2,figsize=(13,9),layout='constrained')
    x = table.component.to_numpy()
    ax = axes[0,0]
    ax.fill_between(x,table.observed_low,table.observed_high,alpha=.2,label='Baseline 5–95% split range')
    ax.plot(x,table.observed_median,'o-',label='Baseline median')
    ax.plot(x,table.effective_threshold,'o--',label=f"{summary['permutation']} {summary['null_quantile']:.0%} threshold (floor 0)")
    ax.set(title='Selection: signed held-out covariance',xlabel='Component rank',ylabel='Covariance');ax.legend(fontsize=8)
    ax=axes[0,1]
    ax.bar(x,table.support,color=['tab:blue' if keep else 'gray' for keep in table.retained])
    ax.axhline(summary['min_support'],color='black',linestyle='--')
    ax.set(title='Selection: fraction of iterations above threshold',xlabel='Component rank',ylabel='Support',ylim=(0,1.05))
    frame=result['components'].query("partition == 'test'")
    ax=axes[1,0]
    for label,color in [('Unpermuted','tab:blue'),('Permuted','tab:orange')]:
        stats=frame[frame.run==label].groupby('component').r
        ranks=stats.median().index
        ax.plot(ranks,stats.median(),'o-',color=color,label=label)
        ax.fill_between(ranks,stats.quantile(.05),stats.quantile(.95),color=color,alpha=.15)
    ax.set(title='Diagnostic: signed correlation (median, 5–95% range)',xlabel='Component rank',ylabel='Pearson r');ax.legend()
    ax=axes[1,1]
    frame=result['metrics'].query("partition == 'test'")
    for j,modality in enumerate(('ieeg','meg')):
        for offset,label,color in [(-.15,'Unpermuted','tab:blue'),(.15,'Permuted','tab:orange')]:
            values=frame.loc[frame.run==label,modality+'_reconstruction_fraction'].dropna()
            if len(values):
                lo,median,hi=np.quantile(values,[.05,.5,.95])
                ax.errorbar(j+offset,median,yerr=[[median-lo],[hi-median]],fmt='o',color=color,
                            label=label if j==0 else None,capsize=4)
    ax.set(xticks=[0,1],xticklabels=['iEEG','MEG'],ylabel='Reconstruction fraction',
           title=f"Diagnostic: reconstruction at evaluated k={summary['n_components']}")
    ax.legend()
    suffix=' (all evaluated ranks supported; evaluate more to locate the boundary)' if summary['boundary_supported'] else ''
    fig.suptitle(f"Stable excess covariance: retain k={summary['selected_k']}{suffix}\n"
                 f"{summary['n_iterations']} matched iterations · {summary['permutation']} · descriptive selection, not a p-value",fontsize=11)
    for ax in fig.axes: ax.grid(True)
    plt.show()